In [3]:
import os
print("현재 작업 위치:", os.getcwd())
print("여기 목록:", os.listdir("."))
print("../data 존재?", os.path.exists("../data"))
print("../data 목록:", os.listdir("../data") if os.path.exists("../data") else "없음")

현재 작업 위치: c:\Users\조영석\ai-study\projects\multihop-agentic-rag\notebooks
여기 목록: ['build_corpus.ipynb', 'chroma_index.ipynb', 'hotpotqa_explore.ipynb']
../data 존재? False
../data 목록: 없음


In [7]:
import json
from sentence_transformers import SentenceTransformer

corpus = json.load(open("..\data\corpus.json", encoding="utf-8"))
titles = list(corpus.keys())
docs   = list(corpus.values())
print("문단 수:", len(docs))

# 비교할 두 모델
model_multi = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
model_eng   = SentenceTransformer("all-MiniLM-L6-v2")

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\조영석\AppData\Local\Temp\ipykernel_32080\1066859808.py:4: SyntaxWarning: invalid escape sequence '\d'
  corpus = json.load(open("..\data\corpus.json", encoding="utf-8"))


문단 수: 4928


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
import chromadb
client = chromadb.Client()          # in-memory (실험용)

def build_collection(name, model):
    col = client.get_or_create_collection(name)
    embs = model.encode(docs, show_progress_bar=True, batch_size=64)
    col.add(ids=titles, documents=docs, embeddings=[e.tolist() for e in embs])
    return col

col_multi = build_collection("multi", model_multi)
col_eng   = build_collection("eng", model_eng)

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

In [12]:
qa = json.load(open("../data/qa.json", encoding="utf-8"))

def search(col, model, query, k=5):
    q = model.encode([query])[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k)
    return res["ids"][0]      # 검색된 문단 제목들

sample = qa[0]
print("Q:", sample["question"])
print("gold:", sample["gold_titles"])          # 정답 근거 (어제 저장한 라벨)
print("multi:", search(col_multi, model_multi, sample["question"]))
print("en   :", search(col_eng,    model_eng,    sample["question"]))

Q: Who was born last, Dave Peverett or Jang Hyun-seung?
gold: ['Dave Peverett', 'Jang Hyun-seung']
multi: ['Kalākaua', 'Jang Hyun-seung', 'Marcos Hernandez (singer)', 'Bang Eun-hee', 'Khun Srun']
en   : ['Jang Kyung-jin', 'Eero Markkanen', 'Jang Hyun-seung', 'Dave Peverett', 'Gus Williams (musician)']


In [15]:
def recall_k(retrieved, gold, k):
    r = set(retrieved[:k])
    g = set(gold)
    return len(g&r)/ len(g)



In [16]:
def eval_recall(col, model, qa, k=5):
    scores = []
    for item in qa:
        q_emb = model.encode([item["question"]])[0].tolist()
        res = col.query(query_embeddings=[q_emb], n_results=k)
        retrieved = res["ids"][0]                    # 상위 k 제목
        scores.append(recall_k(retrieved, item["gold_titles"], k))
    return sum(scores) / len(scores)                 # 500개 평균

print("multi recall@5:", eval_recall(col_multi, model_multi, qa))
print("en    recall@5:", eval_recall(col_eng,    model_eng,    qa))

multi recall@5: 0.57
en    recall@5: 0.739


In [18]:
print("en recall@20:", eval_recall(col_eng, model_eng, qa, k=20))

en recall@20: 0.869


# W1-3 결론
# 임베딩: all-MiniLM-L6-v2 (영어전용) 채택 — recall@5 0.739 vs multi 0.57
# 재정렬 여지: recall@5 0.739 → recall@20 0.869 (13%p gap = rerank가 메울 목표)

In [19]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-base")   # week11에서 쓴 그 모델

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [22]:
def retrieve_and_rerank(query, k_retrieve=20, k_final=5):
    # 1) bi-encoder 검색: 후보 20개 (제목+문단텍스트 둘 다 필요)
    q_emb = model_en.encode([query])[0].tolist()
    res = col_eng.query(query_embeddings=[q_emb], n_results=k_retrieve)
    cand_titles = res["ids"][0]
    cand_docs   = res["documents"][0]

    # 2) cross-encoder 재정렬:
    #    - (query, 문단) 쌍 20개를 만들어 reranker.predict 로 점수 계산
    #    - 점수 내림차순으로 cand_titles 정렬
    #    - 상위 k_final개 제목 반환
    # TODO: 여기 채우기
    pairs = [(query, d) for d in cand_docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores,cand_titles),reverse=True)
    reranked_titles = [title for score, title in ranked]
    return reranked_titles[:k_final]

In [23]:
def eval_rerank(qa, k=5):
    scores = []
    for item in qa:
        retrieved = retrieve_and_rerank(item["question"], k_retrieve=20, k_final=k)
        scores.append(recall_k(retrieved, item["gold_titles"], k))
    return sum(scores) / len(scores)

print("검색만 recall@5   :", 0.739)          # 어제 baseline
print("재정렬후 recall@5 :", eval_rerank(qa, k=5))
print("검색 상한(@20)    :", 0.869)          # 재정렬이 노리는 천장

검색만 recall@5   : 0.739
재정렬후 recall@5 : 0.849
검색 상한(@20)    : 0.869
